# Nowcoder EDA
Existing labels retained. No Reddit data. Full handbook confirmation pending.
Set paths below and run all cells. Dependencies: pandas, openpyxl, matplotlib.

In [ ]:
from pathlib import Path
INPUT_FILE = Path(r"D:/电脑管家迁移文件/xwechat_files/wxid_c2qtxt5mwgat12_ffca/temp/RWTemp/2026-11/0f321e37045139c2e564442e46ebfbd5/Nowcoder_Interview_Include_387.xlsx")
OUTPUT_DIR = Path.cwd() / "nowcoder_eda_results"


In [ ]:
"""Read-only EDA of existing Nowcoder labels; never recodes source data.
Usage: python eda.py --input PATH_TO_XLSX [--output OUTPUT_FOLDER]
Dependencies: pandas openpyxl matplotlib
Existing Guide: 1=explicitly assessed, 0=not reported, U=unclear.
Full handbook was not supplied; no new coding definitions are introduced.
"""
from pathlib import Path
import argparse
import json
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

TOPICS = ['sql', 'programming', 'statistics', 'machine_learning',
          'business_case', 'product_metrics', 'bi_visualization',
          'behavioral', 'project_experience']
LABELS = TOPICS + ['ai_tool_assessment']

def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--input', type=Path, required=True)
    base_dir = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
    parser.add_argument('--output', type=Path, default=base_dir / 'results')
    args = parser.parse_args(['--input', str(INPUT_FILE), '--output', str(OUTPUT_DIR)])
    out = args.output
    out.mkdir(parents=True, exist_ok=True)
    df = pd.read_excel(args.input, sheet_name='Nowcoder_Include', dtype=str, keep_default_na=False)
    required = LABELS + ['record_id', 'url', 'platform', 'relevance_status', 'date']
    missing = sorted(set(required) - set(df.columns))
    if missing:
        raise ValueError(f'Missing required columns: {missing}')
    if set(df.platform.str.strip()) != {'Nowcoder'}:
        raise ValueError('Expected Nowcoder-only input.')
    # Retain existing screening decisions; never silently add or remove records.
    if not df.relevance_status.str.strip().str.lower().eq('include').all():
        raise ValueError('Input contains records not marked include; confirm the intended sample.')
    n = len(df)
    if not n:
        raise ValueError('Empty sample.')
    codes = df[LABELS].apply(lambda c: c.str.strip().str.upper())
    invalid = {c: sorted(set(codes[c]) - {'0', '1', 'U', ''}) for c in LABELS}
    if any(invalid.values()):
        raise ValueError(f'Unexpected label codes: {invalid}')
    records = []
    for c in LABELS:
        counts = codes[c].value_counts()
        yes, no, unclear, blank = [int(counts.get(k, 0)) for k in ['1', '0', 'U', '']]
        records.append(dict(variable=c, explicitly_assessed=yes, not_reported=no,
                            unclear=unclear, blank=blank, total=n,
                            positive_pct_all=100*yes/n,
                            unknown_or_blank_pct=100*(unclear+blank)/n,
                            determinate_n=yes+no,
                            positive_pct_determinate=100*yes/(yes+no) if yes+no else None))
    summary = pd.DataFrame(records)
    assert (summary[['explicitly_assessed','not_reported','unclear','blank']].sum(axis=1) == n).all()
    summary.to_csv(out/'category_summary.csv', index=False, encoding='utf-8-sig')
    dates = pd.to_datetime(df.date, errors='coerce', utc=True)
    years = dates.dt.year.value_counts().sort_index().rename_axis('posting_year').reset_index(name='posts')
    years['pct'] = 100*years.posts/n
    years.to_csv(out/'posting_year.csv', index=False)
    quality = {'n_posts':n, 'unique_record_ids':int(df.record_id.nunique()),
               'unique_urls':int(df.url.nunique()), 'duplicate_id_extra':int(df.record_id.duplicated().sum()),
               'duplicate_url_extra':int(df.url.duplicated().sum()),
               'blank_ids':int(df.record_id.str.strip().eq('').sum()),
               'blank_urls':int(df.url.str.strip().eq('').sum()),
               'invalid_dates':int(dates.isna().sum()),
               'date_min':str(dates.min()), 'date_max':str(dates.max())}
    quality['metadata_counts'] = {c:df[c].value_counts(dropna=False).to_dict() for c in
        ['label_origin','coder_id','text_completeness','assessment_format','rounds_reported','round_count'] if c in df}
    (out/'quality_summary.json').write_text(json.dumps(quality, indent=2, ensure_ascii=False),encoding='utf-8')
    strata = []
    if 'label_origin' in df:
        for origin, idx in df.groupby('label_origin').groups.items():
            for c in LABELS:
                v = codes.loc[idx,c]
                strata.append({'label_origin':origin,'variable':c,'n':len(idx),
                               **{f'code_{k or "blank"}':int(v.eq(k).sum()) for k in ['1','0','U','']}})
    pd.DataFrame(strata).to_csv(out/'labels_by_origin.csv',index=False)
    # Co-occurrence is a count of joint positive labels, not a correlation estimate.
    positives = codes[TOPICS].eq('1').astype(int)
    positives.T.dot(positives).to_csv(out/'positive_cooccurrence_counts.csv')
    plt.rcParams.update({'font.size':10, 'axes.spines.top':False,'axes.spines.right':False})
    plot = summary.set_index('variable').loc[TOPICS].sort_values('positive_pct_all')
    fig, ax = plt.subplots(figsize=(10,5.5), layout='constrained')
    bars = ax.barh(plot.index.str.replace('_',' '),plot.positive_pct_all,color='#286B8C')
    ax.bar_label(bars,labels=[f'{int(r.explicitly_assessed)}/{n} ({r.positive_pct_all:.1f}%)' for _,r in plot.iterrows()],padding=4)
    ax.set_xlim(0,max(25,plot.positive_pct_all.max()+22))
    ax.set_xlabel('Posts with an existing positive label (% of all included posts)')
    ax.set_title('Reported interview categories — existing labels')
    fig.savefig(out/'topic_frequency.png',dpi=180)
    plt.close(fig)
    fig, ax = plt.subplots(figsize=(11,6),layout='constrained')
    comp = summary.set_index('variable')[['explicitly_assessed','not_reported','unclear','blank']]*100/n
    comp.index = comp.index.str.replace('_',' ')
    comp.plot.barh(stacked=True,ax=ax,color=['#286B8C','#CBD5DF','#E6A34B','#777777'])
    ax.set_xlabel('% of all included posts'); ax.set_ylabel('');ax.set_xlim(0,100)
    ax.set_title('Coding completeness and recorded outcomes');ax.legend(loc='lower center',bbox_to_anchor=(.5,1.02),ncol=4,fontsize=8)
    fig.savefig(out/'coding_completeness.png',dpi=180);plt.close(fig)
    top = summary[summary.variable.isin(TOPICS)].sort_values('explicitly_assessed',ascending=False)
    top_text = ', '.join(f"{r.variable.replace('_',' ')} ({int(r.explicitly_assessed)} posts; {r.positive_pct_all:.1f}%)" for _,r in top.head(3).iterrows())
    ai = summary.set_index('variable').loc['ai_tool_assessment']
    year_text = '; '.join(f'{int(r.posting_year)}: {int(r.posts)} ({r.pct:.1f}%)' for _,r in years.iterrows())
    report = f'''# Exploratory Data Analysis

The EDA uses {n} included Nowcoder posts as the unit of analysis. Each post may describe multiple interview rounds and question categories. Existing screening and topic labels were retained without recoding. Category percentages use all {n} posts as the denominator, and unknown or blank labels are reported separately. Categories are not mutually exclusive, so their percentages should not be added together.

## Sample coverage

The dataset contains {quality['unique_record_ids']} unique record identifiers and {quality['unique_urls']} unique URLs. Recorded posting dates range from {dates.min().date()} to {dates.max().date()}, with {quality['invalid_dates']} invalid or missing dates. The posting-year distribution is {year_text}. These dates describe when accounts were posted, rather than necessarily when interviews occurred. The 2026 observations cover only the period through the latest collected post, so annual counts do not establish changes in interview activity.

## Interview topics and question types

Based on the existing positive labels, the three most frequently reported categories are {top_text}. These frequencies describe content recorded in the selected posts. They do not estimate the percentage of all BA/DA interviews testing these skills. A code of 0 means not reported, while U means unclear; neither establishes that a topic was absent from the interview. The categories capture both technical subjects and broader question content rather than a separate, exhaustive taxonomy of question formats.

## AI-tool proficiency

The AI-tool assessment variable contains {int(ai.explicitly_assessed)} positive labels ({ai.positive_pct_all:.1f}% of all posts), {int(ai.not_reported)} not-reported labels ({100*ai.not_reported/n:.1f}%), and {int(ai.unclear)} unclear labels ({100*ai.unclear/n:.1f}%). There are {int(ai.blank)} blank labels. The large unclear share limits conclusions about the frequency of AI-tool assessment. The positive count is an observed coding result requiring evidence review, rather than a validated prevalence estimate. General references to machine learning or AI should not be substituted for evidence of an AI-tool proficiency assessment.

## Interpretation and next steps

The results are descriptive and conditional on the existing coding. Label provenance and text completeness are summarized separately in quality_summary.json and labels_by_origin.csv. Model-predicted and AI-assisted labels require human validation, particularly for AI-tool assessment and ambiguous categories. The next steps are to verify positive evidence, review unclear records against accessible original posts, and check label consistency under the unchanged handbook. Search-based sampling and incomplete excerpts limit generalization. The full handbook was not available during this run; code meanings follow the existing local Guide and must be checked against the original handbook before submission.
'''
    (out/'EDA_analysis_EN.md').write_text(report,encoding='utf-8')
    print(summary.to_string(index=False))
    print(f'Output: {out.resolve()}')

if __name__ == '__main__':
    main()


In [ ]:
from IPython.display import display, Markdown, Image
display(Markdown((OUTPUT_DIR / 'EDA_analysis_EN.md').read_text(encoding='utf-8')))
display(Image(filename=str(OUTPUT_DIR / 'topic_frequency.png')))
display(Image(filename=str(OUTPUT_DIR / 'coding_completeness.png')))
